# 01. Data Ingestion, Cadastral Filtering & Preprocessing

This notebook implements the data ingestion and transformation pipeline for the **Poznań Housing Market Dataset**. It processes transactional cadastral records from the **Rejestr Cen Nieruchomości (RCN)** spanning 2013–2025, applying domain filtering rules to separate arm's-length open-market transactions from institutional/municipal (JST) transfers.

---

### Pipeline Objectives & Methodology
1. **Raw Cadastral Ingestion:** Ingest raw combined transaction records ($N \approx 111,683$) containing transaction metadata, cadastral identifiers, physical property attributes, and geolocation data.
2. **Feature Engineering:** Calculate distance to the city center (`odl_do_centrum`), temporal indicators (`rok_miesiac_float`), macroeconomic joins (`wibor_3m`, `srednie_wynagrodzenie`), and inflation adjustments (`wskaznik_cpi`).
3. **Domain Filtering:**
   - **Property type:** Residential apartments (`lokal mieszkalny`, `funkcja_lokalu == 1.0`).
   - **Transaction type:** Standard open-market sales (`sprzedaz`).
   - **Ownership share:** Full title (`udzial == 1.0`).
   - **Arm's-length condition:** Exclude institutional buyers (`t_kupujacy != 'jednostka_samorzadu'`).
4. **Outlier Mitigation:** Clip extreme target outliers ($\text{PLN/m}^2$) outside the 1st and 99th percentiles to eliminate non-market anomalies while preserving genuine market dispersion.
5. **Output Serialization:** Produce clean datasets for downstream econometric modeling ($N = 38,154$) and municipal institutional analysis ($N = 1,664$).


## 1. Environment Configuration & Module Imports
Set up paths, autoreload extensions, and import domain preprocessing utilities from `src`.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

# Resolve the repo root robustly from either the project root or the notebooks folder.
project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_io import load_data
from src.features import (
    prepare_market_dataframe,
    prepare_jst_dataframe,
    build_residential_mask,
    clip_target_outliers,
)

root = project_root
data_path = root / "data" / "rcn_lokale_polaczone_extra_osiedla.csv"
print(f"project_root={root}")

print(f"data_path_exists={data_path.exists()}")

project_root=/home/gienon/source/repos/ml/projekt
data_path_exists=True


## 2. Cadastral Dataset Ingestion & Feature Engineering
Ingest the raw RCN records and apply base feature extraction: spatial distance to the Poznań Old Market (`odl_do_centrum`), temporal indices, and macroeconomic indicators.


In [2]:
raw = load_data(str(data_path))
df = prepare_market_dataframe(raw)
jst = prepare_jst_dataframe(raw)

print(f"raw rows: {len(raw):,}")
print(f"market rows after original filters: {len(df):,}")
print(f"JST rows after original filters: {len(jst):,}")
print(f"columns: {df.shape[1]}")
df.head()


raw rows: 198,854
market rows after original filters: 111,683
JST rows after original filters: 62,720
columns: 32


,liczba_izb,nr_kondygnacji,calkowita_powierzchnia,wsp_x,wsp_y,adres_miejscowosc,adres_ulica,adres_numer,bud_id,dzialka_id,...,metry_na_izbe,wibor_3m,srednie_wynagrodzenie,is_pierwotny,is_uzytkowanie_wieczyste,sprzedajacy_firma,sprzedajacy_jst,sprzedajacy_osoba,is_wielorodzinny,osiedle_cat
1,2.0,NaN,40.30,5806623.642,6421725.773,Poznań,ul. Sieradzka,22,NaN,NaN,...,18.700000,3.95,3311.0,0,0,1,0,0,0,9
2,2.0,NaN,52.20,5810930.741,6426709.788,Poznań,ul. Starowiejska,1e,NaN,NaN,...,26.100000,3.95,3311.0,0,0,1,0,0,0,27
3,3.0,3.0,55.60,5813369.429,6428381.352,Poznań,ul. Naramowicka,175,NaN,NaN,...,18.533333,3.95,3311.0,0,0,1,0,0,0,15
5,4.0,NaN,55.98,5811714.325,6426461.773,Poznań,os. Powstańców Warszawy,9A,NaN,NaN,...,13.380000,3.95,3311.0,0,0,1,0,0,0,34
6,2.0,5.0,31.60,5807817.792,6429491.534,Poznań,ul. Katowicka,25,NaN,NaN,...,15.800000,3.95,3311.0,0,0,1,0,0,0,24


## 3. Residential Domain Filtering & Target Outlier Clipping
Apply the residential domain mask (`build_residential_mask`) and clip price outliers ($1\text{st} - 99\text{th}$ percentiles) to establish the final modeling sample.


In [3]:
residential_mask = build_residential_mask(df).copy()
df_res = df[residential_mask].copy()
df_res = clip_target_outliers(df_res, target_col="cena_za_m2", lo_q=0.01, hi_q=0.99)

print(f"filtered rows: {len(df_res):,}")
df_res[["osiedle", "cena_za_m2", "powierzchnia_final", "liczba_izb"]].head()


clip_target_outliers: removed 780 rows (2.0%) outside [3972, 17311] PLN/m²
filtered rows: 38,154


,osiedle,cena_za_m2,powierzchnia_final,liczba_izb
3,Naramowice,9348.213932,55.60,3.0
6,Rataje,13621.102064,31.60,2.0
8,Żegrze,8186.101259,50.10,3.0
13,Winiary,10196.694182,90.00,5.0
16,Grunwald Północ,10518.966439,26.25,2.0


## 4. Dataset Serialization
Export the clean residential market dataset (`housing_residential_market.csv`) and municipal institutional dataset (`housing_jst_institutional.csv`) to `data/processed/`.


In [4]:
processed_dir = root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_res.to_csv(processed_dir / "housing_residential_market.csv", index=False)
print(f"Saved market data: {len(df_res):,} rows → {processed_dir / 'housing_residential_market.csv'}")

jst.to_csv(processed_dir / "housing_jst_institutional.csv", index=False)
print(f"Saved JST data: {len(jst):,} rows → {processed_dir / 'housing_jst_institutional.csv'}")

Saved market data: 38,154 rows → /home/gienon/source/repos/ml/projekt/data/processed/housing_residential_market.csv
Saved JST data: 62,720 rows → /home/gienon/source/repos/ml/projekt/data/processed/housing_jst_institutional.csv
